# Notebook 1 — Environment Setup
**Platform: Lambda Labs GPU Cloud (JupyterLab)**

Installs all dependencies on top of Lambda's pre-installed PyTorch environment,
sets the HuggingFace token, clones the required repos, and verifies the GPU.

Run this **once** at the start of a new instance.

**Checklist before proceeding to NB2:**
- GPU shows in `nvidia-smi`
- HF token authenticates successfully
- All imports succeed
- spaCy finds 'slice' as the root verb
- `paths.json` is readable
- DINOv3 repo contains Python files


## Verify GPU


In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


Thu May  7 18:59:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     On  |   00000000:07:00.0 Off |                  Off |
|  0%   28C    P8             10W /  150W |       0MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## HuggingFace Token
Required for:
- Authenticated HF Hub access (avoids rate limits)
- The POS fallback model used in NB3 (`vblagoje/bert-english-uncased-finetuned-pos`)

Paste your token below. It is written to `~/.bashrc` so it persists across sessions
on this instance.


In [2]:
import sys
print(sys.executable)

/home/ubuntu/youcook2-videos/venv/bin/python


In [3]:
import os

# ── Paste your HuggingFace token here ──
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

# Set for this session
os.environ['HF_TOKEN']                = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN']  = HF_TOKEN  # legacy env var

# Persist to ~/.bashrc for future terminal sessions on this instance
bashrc = os.path.expanduser('~/.bashrc')
with open(bashrc, 'a') as f:
    f.write(f'\nexport HF_TOKEN="{HF_TOKEN}"\n')
    f.write(f'export HUGGING_FACE_HUB_TOKEN="{HF_TOKEN}"\n')

# Verify
from huggingface_hub import whoami
try:
    info = whoami(token=HF_TOKEN)
    print(f"HF authenticated as: {info['name']} ✓")
except Exception as e:
    print(f"HF auth failed: {e}")
    print("Check your token and re-run this cell before continuing.")


HF authenticated as: nithura ✓


## Project directory structure
All data, videos, embeddings and checkpoints live under `PROJECT_ROOT`.
Lambda instances have persistent local storage at `/home/ubuntu/` — no cloud sync needed.

`paths.json` is written once here and loaded by every subsequent notebook so
there is exactly one place to update if the root directory ever changes.


In [4]:
import os, json

# Persistent volume root — all project data lives here
PROJECT_ROOT = '/home/ubuntu/youcook2-videos'

PATHS = {
    'root':        PROJECT_ROOT,
    'videos':      os.path.join(PROJECT_ROOT, 'videos'),
    'frames':      os.path.join(PROJECT_ROOT, 'frames'),
    'embeddings':  os.path.join(PROJECT_ROOT, 'embeddings', 'dinov2'),
    'annotations': os.path.join(PROJECT_ROOT, 'annotations'),
    'processed':   os.path.join(PROJECT_ROOT, 'processed'),
    'repos':       os.path.join(PROJECT_ROOT, 'repos'),
    'checkpoints': os.path.join(PROJECT_ROOT, 'checkpoints'),
}

for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

with open(os.path.join(PROJECT_ROOT, 'paths.json'), 'w') as f:
    json.dump(PATHS, f, indent=2)

print("Directory structure ready:")
for k, v in PATHS.items():
    print(f"  {k:15s} → {v}")

Directory structure ready:
  root            → /home/ubuntu/youcook2-videos
  videos          → /home/ubuntu/youcook2-videos/videos
  frames          → /home/ubuntu/youcook2-videos/frames
  embeddings      → /home/ubuntu/youcook2-videos/embeddings/dinov2
  annotations     → /home/ubuntu/youcook2-videos/annotations
  processed       → /home/ubuntu/youcook2-videos/processed
  repos           → /home/ubuntu/youcook2-videos/repos
  checkpoints     → /home/ubuntu/youcook2-videos/checkpoints


## Install packages
Lambda's GPU Base 24.04 image already ships PyTorch with CUDA — we do NOT reinstall it.
We only install the extras that are not pre-loaded.

The `--break-system-packages` flag is required on Ubuntu 24.04 to install into
the system Python that JupyterLab uses, without setting up a virtual environment.


In [5]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([
        '/lambda/nfs/youcook2-videos/venv/bin/pip',
        'install', '-q', *args
    ])

import torch
print(f"PyTorch already installed: {torch.__version__}  CUDA: {torch.version.cuda}")

pip('timm', 'einops')
pip('transformers', 'datasets', 'accelerate', 'huggingface_hub')
pip('spacy')
# Install spaCy model directly from wheel — more reliable than `python -m spacy download`
pip('https://github.com/explosion/spacy-models/releases/download/'
    'en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl')
pip('decord')
pip('yt-dlp')
pip('pandas', 'tqdm', 'Pillow', 'opencv-python-headless')
pip('scikit-learn', 'matplotlib', 'seaborn')

print("\nAll extras installed.")


PyTorch already installed: 2.11.0+cu130  CUDA: 13.0


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.2.5 requires numpy<2.0.0,>=1.19.0; python_version >= "3.9", but you have numpy 2.4.4 which is incompatible.



All extras installed.


## Clone Facebook Research repos
We clone `dinov2` (contains DinoTXT), `vjepa2`, and `dinov3`.
They are NOT pip-installed — we add them to `sys.path` in the relevant notebooks
to keep the environment clean.

**Note on DINOv3:** This repo may not be fully public yet. The verification cell
below checks its contents — if it is empty, confirm the correct URL with your supervisor
before running NB4+.


In [8]:
import os, subprocess

REPOS = PATHS['repos']

def clone_or_pull(url, name):
    dest = os.path.join(REPOS, name)
    if os.path.exists(os.path.join(dest, '.git')):
        print(f"{name}: already cloned, pulling latest...")
        subprocess.run(['git', '-C', dest, 'pull', '--quiet'], check=True)
    else:
        print(f"{name}: cloning...")
        subprocess.run(['git', 'clone', '--quiet', url, dest], check=True)
    print(f"  {dest} ✓")

clone_or_pull('https://github.com/facebookresearch/dinov2.git', 'dinov2')
clone_or_pull('https://github.com/facebookresearch/vjepa2.git',  'vjepa2')
clone_or_pull('https://github.com/facebookresearch/dinov3.git',  'dinov3')


dinov2: already cloned, pulling latest...
  /home/ubuntu/youcook2-videos/repos/dinov2 ✓
vjepa2: already cloned, pulling latest...
  /home/ubuntu/youcook2-videos/repos/vjepa2 ✓
dinov3: already cloned, pulling latest...
  /home/ubuntu/youcook2-videos/repos/dinov3 ✓


## Verify all imports


In [6]:
import sys, json, os
import torch, torchvision, transformers
import spacy, decord, timm, einops
import pandas as pd
from PIL import Image

print(f"PyTorch      {torch.__version__}")
print(f"CUDA         {torch.version.cuda}")
print(f"GPU          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '*** NOT AVAILABLE ***'}")
print(f"Transformers {transformers.__version__}")
print(f"timm         {timm.__version__}")
print(f"decord       {decord.__version__}")

# spaCy verb extraction test
nlp  = spacy.load('en_core_web_sm')
doc  = nlp("slice the onions finely")
root = [t for t in doc if t.dep_ == 'ROOT']
print(f"\nspaCy root verb test: '{root[0].text}' ✓" if root else "\nspaCy: no ROOT found ✗")

# paths.json readable
with open(os.path.join(PATHS['root'], 'paths.json')) as f:
    test = json.load(f)
print(f"paths.json readable ✓  ({len(test)} entries)")

# HF token still set
hf_tok = os.environ.get('HF_TOKEN', '')
print(f"HF_TOKEN set: {'✓' if hf_tok.startswith('hf_') else '✗ — re-run the token cell'}")


PyTorch      2.11.0+cu130
CUDA         13.0
GPU          NVIDIA A10
Transformers 5.8.0
timm         1.0.26
decord       0.6.0


/home/ubuntu/youcook2-videos/venv/lib/python3.12/site-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)



spaCy root verb test: 'slice' ✓
paths.json readable ✓  (8 entries)
HF_TOKEN set: ✓


## Done ✓
Environment is ready. Proceed to **Notebook 2**.
